# 🔄 Re-entrenamiento con k=8 Clusters (ÓPTIMO)

**Objetivo**: Re-entrenar el modelo con k=8 clusters (resultado óptimo del Notebook 1)

**Output**:
- Modelo KMeans con 8 clusters
- Mismo PCA del modelo original
- Guardado en `../models/kmeans_k8.pkl`

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

## 1. Cargar Datos

In [2]:
# Cargar datos
df = pd.read_csv('../../clustering/data/df_for_clustering.csv')
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df = df[df['deceased'] == 'N']

print(f"Datos cargados: {df.shape}")
print(f"Clientes: {len(df):,}")

Datos cargados: (442909, 23)
Clientes: 442,909


## 2. Preprocesamiento (IGUAL que modelo original)

In [3]:
# Filtrar y limpiar
df.drop(columns=['deceased', 'gender', 'entry_channel'], inplace=True, errors='ignore')
df.drop(columns=['z_pct_months_active', 'account', 'z_num_families'], inplace=True, errors='ignore')
df.drop(columns=['macro_region', 'is_coast'], inplace=True, errors='ignore')

# Guardar IDs
df_ids = df[['pk_cid', 'pk_partition']].copy()
df.drop(columns=['pk_cid', 'pk_partition'], inplace=True)

print(f"Features después de limpieza: {df.shape[1]}")

Features después de limpieza: 13


In [4]:
# One-Hot Encoding
def one_hot_encode(df, features):
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    encoded = encoder.fit_transform(df[features])
    encoded_df = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out(features),
        index=df.index
    )
    df = df.drop(columns=features)
    return pd.concat([df, encoded_df], axis=1)

df = one_hot_encode(df, ['segment', 'city_size'])
print(f"Después de OHE: {df.shape}")

Después de OHE: (442909, 18)


In [5]:
# Normalización
scaler = MinMaxScaler()
df[['age', 'salary', 'z_months_since_entry', 'z_num_products']] = scaler.fit_transform(
    df[['age', 'salary', 'z_months_since_entry', 'z_num_products']]
)

print("✅ Normalización completada")

✅ Normalización completada


## 3. Cargar PCA Original

In [6]:
# Usar el mismo PCA del modelo original para comparabilidad
pca = joblib.load('../../clustering/models/pca_model.pkl')

print(f"PCA cargado: {pca.n_components_} componentes")
print(f"Varianza explicada: {pca.explained_variance_ratio_.sum():.4f}")

X_pca = pca.transform(df)
print(f"Datos transformados: {X_pca.shape}")

PCA cargado: 6 componentes
Varianza explicada: 0.9136
Datos transformados: (442909, 6)


## 4. Entrenar K-Means con k=8

In [7]:
print("Entrenando K-Means con k=8...")
print("Esto puede tardar 3-5 minutos...\n")

kmeans_k8 = KMeans(n_clusters=8, n_init=10, random_state=42)
kmeans_k8.fit(X_pca)

print("✅ Entrenamiento completado")
print(f"Inertia: {kmeans_k8.inertia_:.2f}")

Entrenando K-Means con k=8...
Esto puede tardar 3-5 minutos...

✅ Entrenamiento completado
Inertia: 154943.79


## 5. Métricas del Modelo k=8

In [8]:
labels_k8 = kmeans_k8.predict(X_pca)

sil_k8 = silhouette_score(X_pca, labels_k8)
db_k8 = davies_bouldin_score(X_pca, labels_k8)
ch_k8 = calinski_harabasz_score(X_pca, labels_k8)

print("="*60)
print("MÉTRICAS DEL MODELO OPTIMIZADO (k=8)")
print("="*60)
print(f"Silhouette Score:      {sil_k8:.4f}  (mayor mejor)")
print(f"Davies-Bouldin Index:  {db_k8:.4f}  (menor mejor)")
print(f"Calinski-Harabasz:     {ch_k8:.2f}  (mayor mejor)")
print(f"Inertia:               {kmeans_k8.inertia_:.2f}")
print("="*60)

MÉTRICAS DEL MODELO OPTIMIZADO (k=8)
Silhouette Score:      0.5436  (mayor mejor)
Davies-Bouldin Index:  0.9124  (menor mejor)
Calinski-Harabasz:     219499.47  (mayor mejor)
Inertia:               154943.79


## 6. Distribución de Clusters

In [9]:
clusters = labels_k8 + 1  # +1 para que empiecen en 1

print("\n📊 DISTRIBUCIÓN DE 8 CLUSTERS\n")
dist = pd.Series(clusters).value_counts().sort_index()
for k, count in dist.items():
    pct = (count / len(clusters)) * 100
    print(f"Cluster {k}: {count:6,} clientes ({pct:5.1f}%)")

print(f"\nTotal: {len(clusters):,} clientes")


📊 DISTRIBUCIÓN DE 8 CLUSTERS

Cluster 1: 49,104 clientes ( 11.1%)
Cluster 2: 79,071 clientes ( 17.9%)
Cluster 3: 48,934 clientes ( 11.0%)
Cluster 4: 52,933 clientes ( 12.0%)
Cluster 5: 81,893 clientes ( 18.5%)
Cluster 6: 45,095 clientes ( 10.2%)
Cluster 7: 47,953 clientes ( 10.8%)
Cluster 8: 37,926 clientes (  8.6%)

Total: 442,909 clientes


## 7. Guardar Modelo

In [10]:
# Guardar modelo k=8
joblib.dump(kmeans_k8, '../models/kmeans_k8.pkl')

print("✅ Modelo guardado: ../models/kmeans_k8.pkl")
print("\n🎯 Próximo paso: Ejecutar notebooks 02, 03, 04 con versión _k8")

✅ Modelo guardado: ../models/kmeans_k8.pkl

🎯 Próximo paso: Ejecutar notebooks 02, 03, 04 con versión _k8
